# RAG Generation Layer
### Ein geführter Workshop zur fünften Online-Stufe: von echten Retrieval-Ergebnissen zu geprüften Antworten

## Anschluss an die Retrieval Layer

Die Retrieval Layer ist abgeschlossen. Ihr Ergebnis ist pro Query eine geordnete Liste von
`RetrievalResult`-Objekten: Dokumentfragmente mit Text, Retrieval-Score und Herkunft
(dense/sparse/fused). Die Generation Layer nimmt genau diese Liste entgegen und verarbeitet
sie in vier Schritten:

1. `ContextPreparer`: Normalisierung, Deduplizierung und Begrenzung auf ein Zeichen-Budget
2. `PromptBuilder`: Einsetzen der Chunks in ein Template
3. `OllamaClient`: HTTP-POST an den Ollama-Endpunkt `/api/generate`
4. `GenerationResult`: Antwort, Latenz und alle Messwerte als strukturiertes Ergebnis

Generation ist die fünfte und letzte Stufe der Pipeline und vollständig online.

### Architektonischer Überblick: alle fünf Stufen

| Layer | Betrieb | Ausgabe |
|---|---|---|
| Ingestion | offline | `chunks.jsonl` |
| Embedding | offline | `embeddings.jsonl` |
| Indexing | offline | `DenseIndex`, `SparseIndex`, `DocumentStore` |
| Retrieval | online | `List[RetrievalResult]` pro Query |
| **Generation** | **online** | **`GenerationResult` pro Query** |

Die Offline-Stufen sind Investitionen: einmalig teuer, danach wiederverwendbar. Die beiden
Online-Stufen sind der Laufzeitpfad für jede Nutzerfrage.

### Was dieses Notebook zeigt

Nicht die Bedienung einer API, sondern das Systemverhalten unter realen Bedingungen:

1. wie echte Retrieval-Ergebnisse in Kontext übersetzt werden und wo dabei Fehler entstehen
2. warum Retrieval-Ranking oft wichtiger ist als das Sprachmodell
3. wie das Zeichen-Budget wirkt und welche Chunks es verwirft
4. wie Templates das Modellverhalten steuern
5. was Temperature mit Reproduzierbarkeit und Halluzination macht
6. wann SimpleRAG ausreicht und wann RefineRAG sinnvoll ist
7. wo diese Architektur bewusste Grenzen zieht

### Artefakt-Abhängigkeit

Dieses Notebook liest ausschließlich persistierte Artefakte aus den vorherigen Notebooks. Es
erzeugt keine Demo-Daten als Fallback. Voraussetzungen sind das Indexing-Notebook mit
`persist=True` (Abschnitt 16), das Embedding-Modell (BAAI/bge-m3) lokal verfügbar und Ollama
mit Mistral unter `http://ollama:11434`. Fehlen Artefakte, bricht das Notebook mit einer klaren
Fehlermeldung ab.


## Welche Bausteine dieses Notebook verwendet

Die nächste Zelle importiert alles und setzt Logging auf `WARNING` (für Debugging, etwa
Retry-Versuche, HTTP-Details und Budget-Kürzungen, genügt `logging.DEBUG` auf
`rag.generation`). Überblick über die Module und ihre Rolle:

**Generation Layer (`rag.generation`).**
`GenerationConfig` (eingefrorene Konfiguration), `OllamaClient` (HTTP-Transport zum lokalen
Modell), `ContextPreparer` (Kontext aufbereiten und budgetieren), `PromptBuilder` und
`PromptTemplate` (Prompt-Erstellung), `SimpleRAGStrategy` und `RefineRAGStrategy` (die beiden
Generierungs-Strategien), die fertigen Templates `STRICT_RAG_TEMPLATE`, `COT_TEMPLATE` und
`FEW_SHOT_TEMPLATE`, das Ergebnisobjekt `GenerationResult`, die Fehlerklassen
(`GenerationError`, `LLMConnectionError`, `LLMTimeoutError`, `LLMResponseError`) sowie
`format_result_summary` für eine kompakte Ausgabe.

**Retrieval und Indexing (für echte Eingaben).**
`rag.indexing.orchestrator.IndexingOrchestrator` und `rag.indexing.config` laden die Indizes
und den DocumentStore. `rag.indexing.sparse.tokenizer.Tokenizer` liefert die Query-Tokens.
`rag.retrieval.orchestrator.RetrievalOrchestrator` und `rag.retrieval.config` erzeugen die
echten Retrieval-Ergebnisse, die diese Stufe als Eingabe bekommt.

**Embedding Layer (für Query-Vektoren).**
`rag.embedding.config`, `rag.embedding.factory.create_embedder` und
`rag.embedding.model_cache.get_default_cache` bauen den Embedder für Dense-Queries, mit
derselben Konfiguration wie beim Index-Aufbau.

Dazu `logging`, `time`, `math` und `pathlib` aus der Standardbibliothek. Die wichtigsten
Stellschrauben für eigene Versuche liegen in `GenerationConfig` (Temperature, Token-Limit,
Zeichen-Budget), in der Wahl des Templates und in der Wahl der Strategie. Jede dieser
Komponenten lässt sich einzeln austauschen, weil die Strategie ihre Bausteine von außen
übergeben bekommt.


In [ ]:
import logging
import time
import math
from pathlib import Path

# Alle RAG-Submodule auf WARNING; Generation-Schicht bei Bedarf auf DEBUG setzen.
for logger_name in ["rag.generation", "rag.retrieval", "rag.indexing", "rag.embedding"]:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

logging.basicConfig(level=logging.WARNING)

# --- Generation Layer ---
from rag.generation import (
    GenerationConfig,
    OllamaClient,
    ContextPreparer,
    PromptBuilder,
    PromptTemplate,
    SimpleRAGStrategy,
    RefineRAGStrategy,
    STRICT_RAG_TEMPLATE,
    COT_TEMPLATE,
    FEW_SHOT_TEMPLATE,
    GenerationResult,
    GenerationError,
    LLMConnectionError,
    LLMTimeoutError,
    LLMResponseError,
    format_result_summary,
)

# --- Retrieval und Indexing Layer (für echte Retrieval-Ergebnisse) ---
from rag.indexing.orchestrator import IndexingOrchestrator
from rag.indexing.config import IndexConfig, DenseIndexConfig, SparseIndexConfig
from rag.indexing.sparse.tokenizer import Tokenizer
from rag.retrieval.orchestrator import RetrievalOrchestrator
from rag.retrieval.config import (
    RetrievalConfig, BM25Config, DenseRetrievalConfig,
    FusionConfig, RerankerConfig,
)
from rag.retrieval.types import RetrievalTrace

# --- Embedding Layer (für Query-Vektoren) ---
from rag.embedding.config import (
    EmbeddingConfig, EmbeddingBehaviorConfig, EmbeddingProjectionConfig
)
from rag.embedding.factory import create_embedder
from rag.embedding.model_cache import get_default_cache

print("Imports abgeschlossen.")
print("Ollama-Verbindung wird in Abschnitt 2 geprüft.")
print("Retrieval-Artefakte werden in Abschnitt 1 geladen.")

---

## 1. Persistierte Artefakte laden

Die Generation Layer hat keinen eigenen Offline-Schritt. Sie liest vollständig
fertige Artefakte aus der Indexing Layer. Dieser Abschnitt lädt sie und prüft
ihre Vollständigkeit.

Das Notebook bricht mit einer klaren Fehlermeldung ab, wenn Artefakte fehlen.
Es gibt keinen automatischen Fallback auf synthetische Daten.

In [ ]:
index_base   = Path("/home/jovyan/workspace/data/index")
persist_path = index_base / "persist_demo"

required_paths = {
    "persist_demo/":    persist_path,
    "documents.jsonl":  persist_path / "documents.jsonl",
}

print("Artefakt-Prüfung:")
all_present = True
for label, path in required_paths.items():
    exists = path.exists()
    if exists and path.is_file():
        size_str = f"{path.stat().st_size / 1024:.1f} KB"
    elif exists:
        size_str = "dir"
    else:
        size_str = "FEHLT"
    status = "OK  " if exists else "FAIL"
    print(f"  [{status}]  {label:<28}  {size_str}")
    if not exists:
        all_present = False

if not all_present:
    raise RuntimeError(
        "Indexing-Artefakte fehlen. "
        "Bitte zuerst das Indexing-Notebook vollständig ausführen (Abschnitt 16, persist=True). "
        "Dieses Notebook erzeugt keine Demo-Daten."
    )

print()
print("Alle Artefakte vorhanden.")

In [ ]:
# Index laden: DenseIndex + SparseIndex + DocumentStore
cfg = IndexConfig(
    index_dir=persist_path,
    mode="hybrid",
    dense=DenseIndexConfig(backend="brute_force", metric="cosine"),
    sparse=SparseIndexConfig(tokenizer="simple"),
)

index_result = IndexingOrchestrator(cfg).load()

dense_index  = index_result.dense_index
sparse_index = index_result.sparse_index
doc_store    = index_result.document_store

all_docs  = list(doc_store.stream_all())
doc_by_id = doc_store.load_by_id()

print(f"DenseIndex     : {type(dense_index).__name__}")
print(f"SparseIndex    : {type(sparse_index).__name__}")
print(f"DocumentStore  : {len(all_docs)} Dokumente")
print()

# Dimensionscheck
docs_with_dense = [d for d in all_docs if d.get("dense_vector")]
if not docs_with_dense:
    raise RuntimeError(
        "DocumentStore enthält keine Dokumente mit dense_vector. "
        "Das Indexing-Notebook muss im Hybrid- oder Dense-Modus ausgeführt worden sein."
    )

dense_dims = {len(d["dense_vector"]) for d in docs_with_dense}
if len(dense_dims) != 1:
    raise RuntimeError(f"Inkonsistente Dense-Dimensionen im DocumentStore: {dense_dims}")

DENSE_DIM = next(iter(dense_dims))
print(f"Dense-Dimension : {DENSE_DIM}")
print(f"Dokumente (dense): {len(docs_with_dense)} / {len(all_docs)}")

---

## 2. Embedder und Retrieval-Orchestrator konfigurieren

Die Generation Layer steuert selbst keine Retrieval-Logik. Sie empfängt fertige
`RetrievalResult`-Objekte. Damit wir in diesem Notebook echte Ergebnisse sehen,
bauen wir den Retrieval-Orchestrator auf, und zwar exakt mit der Konfiguration aus dem
Retrieval-Notebook.

**Architektonische Anforderung:** Der Embedder muss dieselbe Konfiguration verwenden
wie beim Index-Aufbau. Insbesondere gleiche Modell-Klasse und gleiche Output-Dimension.
Ein Mismatch führt sofort zu `ValueError`.

In [ ]:
# Embedder laden, identische Konfiguration zum Embedding-Notebook
embedding_config = EmbeddingConfig(
    provider="bge-m3",
    model_name="BAAI/bge-m3",
    model_version="1.0",
    device="cpu",
    batch_size=1,
    use_fp16=False,
    retrieval_mode="dense",
    behavior=EmbeddingBehaviorConfig(normalize=True, mode="query"),
    projection=EmbeddingProjectionConfig(target_dim=None, method="truncate", model_aware=True),
)

EMBEDDER_AVAILABLE = False
embedder = None

try:
    embedder = create_embedder(embedding_config, cache=get_default_cache())
    EMBEDDER_AVAILABLE = True
    print(f"Embedder geladen : {embedder.model_type()}")
    print(f"Dimension        : {embedder.dimension()}")
    # Dimensionscheck Embedder <-> Index
    if embedder.dimension() is not None and embedder.dimension() != DENSE_DIM:
        raise RuntimeError(
            f"Dimensionsinkonsistenz: Embedder={embedder.dimension()}, "
            f"DenseIndex={DENSE_DIM}. Embedding-Konfiguration prüfen."
        )
    print(f"Dimensionskonsistenz: Embedder ({embedder.dimension()}) = DENSE_DIM ({DENSE_DIM})")
except Exception as exc:
    print(f"Embedder nicht verfügbar: {exc}")
    print()
    print("Nur Sparse-Retrieval (BM25) steht ohne Embedder zur Verfügung.")
    print("Hybrid- und Dense-Retrieval-Abschnitte werden übersprungen.")

In [ ]:
# Tokenizer und Retrieval-Orchestrator konfigurieren
tokenizer = Tokenizer(SparseIndexConfig(tokenizer="simple"))

view = sparse_index.get_view()
stats = view.get_stats()
print(f"Vocabulary     : {len(view.get_vocabulary())} Terme")
print(f"doc_count      : {stats['doc_count']}")
print(f"avg_doc_length : {stats['avg_doc_length']:.2f} Tokens")
print()

# Retrieval-Config: Hybrid wenn Embedder verfügbar, sonst BM25-only
if EMBEDDER_AVAILABLE:
    retrieval_mode = "hybrid"
    retrieval_config = RetrievalConfig(
        mode="hybrid",
        sparse_strategy="bm25",
        top_k=5,
        dense=DenseRetrievalConfig(candidate_k=15),
        bm25=BM25Config(k1=1.5, b=0.75, candidate_k=15),
        fusion=FusionConfig(
            strategy="weighted_sum",
            dense_weight=0.5,
            sparse_weight=0.5,
            normalize_scores=True,
            combination="union",
        ),
        reranker=RerankerConfig(enabled=False),
    )
    retrieval_orch = RetrievalOrchestrator(
        document_store=doc_store,
        config=retrieval_config,
        embedder=embedder,
        dense_index=dense_index,
        sparse_index=sparse_index,
        tokenizer=tokenizer,
    )
else:
    retrieval_mode = "sparse_bm25"
    retrieval_config = RetrievalConfig(
        mode="sparse_bm25",
        top_k=5,
        bm25=BM25Config(k1=1.5, b=0.75, candidate_k=15),
        reranker=RerankerConfig(enabled=False),
    )
    retrieval_orch = RetrievalOrchestrator(
        document_store=doc_store,
        config=retrieval_config,
        sparse_index=sparse_index,
        tokenizer=tokenizer,
    )

print(f"Retrieval-Modus : {retrieval_mode}")
print(f"Orchestrator konfiguriert.")

---

## 3. Erste echte Retrieval-Ergebnisse

Bevor die Generation-Schicht eine einzige Zeile Code sieht, ist es wichtig zu
verstehen, was sie bekommt. Dieser Abschnitt führt echte Retrieval-Abfragen aus
und inspiziert die Ergebnisse systematisch.

Das ist kein Aufwärmschritt. Es ist der wichtigste Schritt des gesamten Notebooks:
die Eingabe der Generation-Schicht ist identisch mit der Ausgabe der Retrieval-Schicht,
und die Qualität dieser Eingabe bestimmt fast vollständig die Qualität der Antwort.

In [ ]:
# Erste Query: echte Retrieval-Ergebnisse inspizieren
erste_query = "Wie funktioniert Retrieval Augmented Generation?"

trace: RetrievalTrace = retrieval_orch.retrieve_with_trace(query=erste_query, k=5)

print(f"Query           : '{erste_query}'")
print(f"Retrieval-Modus : {trace['mode']}")
if trace.get("query_tokens"):
    print(f"Query-Tokens    : {trace['query_tokens']}")
print(f"Dense Kandidaten: {len(trace['dense_candidates'])}")
print(f"Sparse Kandidaten: {len(trace['sparse_candidates'])}")
print(f"Finale Ergebnisse: {len(trace['final_results'])}")
print()

retrieval_results = trace["final_results"]

In [ ]:
# Retrieval-Ergebnisse vollständig inspizieren
print("Top Retrieval-Ergebnisse:")
print()

for i, result in enumerate(retrieval_results):
    score_info = f"score={result['score']:.5f}"
    if result.get("dense_score") is not None:
        score_info += f"  dense={result['dense_score']:+.4f}"
    if result.get("sparse_score") is not None:
        score_info += f"  sparse={result['sparse_score']:.4f}"

    print(f"[{i}] {score_info}")
    print(f"    chunk_id    : {result.get('chunk_id', 'n/a')}")
    print(f"    document_id : {result.get('document_id', 'n/a')}")

    text_preview = result["text"][:200].replace("\n", " ")
    print(f"    text[:200]  : {text_preview!r}")
    print()

### Was diese Scores bedeuten

Der Fused Score ist das Ergebnis der Weighted-Sum-Fusion von normalisierten Dense- und
Sparse-Scores. Er sagt: wie relevant ist dieser Chunk für die Query, nach Meinung des
Retrieval-Systems.

Was er nicht sagt: ob der Chunk tatsächlich nützlich für die Beantwortung ist.
Das ist ein fundamentaler Unterschied, der in diesem Notebook systematisch untersucht wird.

**Dense-Score** misst semantische Ähnlichkeit: ähnliche Bedeutung, ähnlicher Kontext.
**Sparse-Score** (BM25) misst lexikalische Überlappung: gleiche oder verwandte Terme.

Ein hoher fused Score garantiert weder, dass der Chunk die Frage beantwortet, noch
dass er präzise genug ist. Retrieval-Qualität ist eine notwendige, keine hinreichende
Bedingung für gute Generierung.

In [ ]:
# Retrieval-Analyse: Kandidaten-Überschneidung Dense/Sparse
if EMBEDDER_AVAILABLE:
    dense_ids  = {c["id"] for c in trace["dense_candidates"]}
    sparse_ids = {c["id"] for c in trace["sparse_candidates"]}

    only_dense  = dense_ids - sparse_ids
    only_sparse = sparse_ids - dense_ids
    both        = dense_ids & sparse_ids

    print(f"Nur Dense gefunden      : {len(only_dense)}")
    print(f"Nur Sparse gefunden     : {len(only_sparse)}")
    print(f"Von beiden Legs gefunden: {len(both)}")
    print()
    print("Diese Zahlen zeigen, ob Dense und Sparse komplementäre Dokumente finden.")
    print("Wenn die Überschneidung gering ist, ist Hybrid-Retrieval sinnvoll.")
    print("Wenn beide Legs dasselbe finden, ist der Hybrid-Mehrwert minimal.")
else:
    print("Nur Sparse-Retrieval: keine Dense/Sparse-Analyse möglich.")

---

## 4. Chunk-Qualität analysieren: Was wurde abgerufen und warum?

Retrieval-Ergebnisse kommen nicht aus dem Nichts. Sie sind das Ergebnis von
Chunking-Entscheidungen aus der Ingestion Layer, Embedding-Qualität aus der
Embedding Layer und Index-Aufbau aus der Indexing Layer.

Dieser Abschnitt untersucht, welche Eigenschaften die abgerufenen Chunks haben
und welche Probleme dabei sichtbar werden.

In [ ]:
# Chunk-Eigenschaften der Retrieval-Ergebnisse messen
print("Chunk-Statistiken der retrieved Dokumente:")
print()

chunk_lengths = [len(r["text"]) for r in retrieval_results]
token_estimates = [len(r["text"].split()) for r in retrieval_results]

print(f"{'Rang':<5} {'Chars':>6} {'Wörter (ca.)':>13} {'Score':>8}  Vorschau")
print("-" * 72)
for i, result in enumerate(retrieval_results):
    text_len  = len(result["text"])
    word_est  = len(result["text"].split())
    score     = result["score"]
    preview   = result["text"][:50].replace("\n", " ")
    print(f"  {i:<3}  {text_len:>6}  {word_est:>13}  {score:>8.5f}  {preview!r}...")

print()
if chunk_lengths:
    print(f"Durchschnittliche Chunk-Länge : {sum(chunk_lengths)/len(chunk_lengths):.0f} Zeichen")
    print(f"Minimale Chunk-Länge          : {min(chunk_lengths)} Zeichen")
    print(f"Maximale Chunk-Länge          : {max(chunk_lengths)} Zeichen")

In [ ]:
# Chunk-Probleme sichtbar machen
# In echten Pipelines treten häufig diese Probleme auf:

probleme_gefunden = []

for i, result in enumerate(retrieval_results):
    text = result["text"]

    # Problem 1: Sehr kurze Chunks (weniger als 100 Zeichen)
    if len(text) < 100:
        probleme_gefunden.append(
            f"  [Rang {i}] Sehr kurzer Chunk: {len(text)} Zeichen, "
            "möglicherweise unvollständiger Satz oder Metadaten-Fragment"
        )

    # Problem 2: Chunk beginnt mit Satzzeichen oder Binding-Wörtern
    stripped = text.strip()
    binding_starts = ("und ", "oder ", "aber ", "however ", "therefore ", "thus ", "furthermore ")
    if any(stripped.lower().startswith(w) for w in binding_starts):
        probleme_gefunden.append(
            f"  [Rang {i}] Chunk beginnt mit Bindewort, "
            "Chunking hat möglicherweise mitten im Satz getrennt"
        )

    # Problem 3: Hohe Textüberlappung mit einem anderen retrieved Chunk
    for j, other in enumerate(retrieval_results):
        if i >= j:
            continue
        overlap = len(set(text.split()) & set(other["text"].split()))
        total   = len(set(text.split()) | set(other["text"].split()))
        if total > 0 and overlap / total > 0.8:
            probleme_gefunden.append(
                f"  [Rang {i}+{j}] Hohe Überlappung ({overlap/total:.0%}), "
                "möglicherweise überlappende Chunks aus der Ingestion"
            )

if probleme_gefunden:
    print("Chunk-Qualitätsprobleme in den Retrieval-Ergebnissen:")
    for p in probleme_gefunden:
        print(p)
else:
    print("Keine offensichtlichen Chunk-Qualitätsprobleme gefunden.")
    print()
    print("Das bedeutet nicht, dass die Chunks perfekt sind, nur dass die")
    print("einfachen heuristischen Tests sie nicht als problematisch einstufen.")

print()
print("Wichtiger Zusammenhang: Chunking-Entscheidungen aus der Ingestion Layer")
print("wirken hier direkt. Wer die Chunks teilt, entscheidet, was Retrieval")
print("überhaupt finden kann. Das ist keine Optimierungsaufgabe der Generation Layer.")

---

## 5. GenerationConfig: Warum Konfiguration unveränderlich ist

Alle Experiment-Parameter der Generation-Schicht leben in `GenerationConfig`.
Das Objekt ist `frozen=True`: einmal erzeugt, kann es nicht verändert werden.

Das löst ein konkretes Problem: In explorativen Experimenten wird eine Variable
irgendwo in der Session geändert, und man weiß später nicht mehr, mit welcher
Konfiguration ein Ergebnis erzeugt wurde. `frozen=True` erzwingt, dass jede
Variante ein separates, eindeutiges Objekt ist.

`GenerationResult` speichert alle Parameter als Momentaufnahme. Das Ergebnis ist
vollständig reproduzierbar, ohne Kontext aus der Session zu benötigen.

In [ ]:
# Basis-Konfiguration für dieses Notebook
base_config = GenerationConfig(
    model_name="mistral",
    temperature=0.0,       # greedy decoding, deterministisch und reproduzierbar
    max_tokens=512,        # num_predict im Ollama-Schema
    seed=42,               # RNG-Seed; wirksam bei temperature=0.0
    top_p=1.0,             # kein Nucleus-Sampling-Filter
    repeat_penalty=1.1,    # leichte Wiederholungsstrafe
    timeout=120.0,         # CPU-Inferenz kann für lange Prompts mehrere Minuten dauern
    max_retries=3,         # Wiederholungsversuche bei Verbindungsfehlern
    retry_delay=2.0,       # feste Wartezeit zwischen Versuchen
    max_context_chars=4000, # Zeichen-Budget für assemblierten Kontext
)

print(f"Modell     : {base_config.model_name}")
print(f"Ollama-URL : {base_config.url}")
print(f"Options    : {base_config.ollama_options}")
print()

# frozen=True demonstrieren
try:
    base_config.temperature = 0.5
except Exception as e:
    print(f"Mutation abgefangen: {type(e).__name__}")
    print("Erwartet. Für eine andere Temperature muss ein neues Objekt erzeugt werden.")

In [ ]:
# Validierung in __post_init__: ungültige Parameter scheitern beim Erstellen, nicht beim Aufruf
ungueltige_configs = [
    ("temperature > 2.0",  dict(model_name="mistral", temperature=3.0)),
    ("max_tokens = 0",     dict(model_name="mistral", max_tokens=0)),
    ("leerer model_name",  dict(model_name="")),
    ("top_p = 0.0",        dict(model_name="mistral", top_p=0.0)),
    ("max_context_chars=0", dict(model_name="mistral", max_context_chars=0)),
]

print("Validierung ungültiger Parameter:")
for label, params in ungueltige_configs:
    try:
        GenerationConfig(**params)
        print(f"  {label}: kein Fehler (unerwartet)")
    except ValueError as e:
        print(f"  {label}: ValueError korrekt")

print()
print("Der Vorteil: Konfigurationsfehler fallen sofort auf, nicht erst")
print("wenn nach 90 Sekunden CPU-Inferenz ein Timeout-Fehler auftritt.")

---

## 6. OllamaClient: HTTP-Transport und Fehlerbehandlung

`OllamaClient` hat eine einzige Verantwortlichkeit: einen HTTP-POST an den
Ollama-Endpunkt schicken und die Antwort validiert zurückgeben.

### Exception-Hierarchie

```text
GenerationError              Basisklasse: alles aus der Generation-Schicht fangen
  LLMConnectionError         TCP-Verbindung fehlgeschlagen, Ollama nicht erreichbar
  LLMTimeoutError            Timeout, Ollama antwortet nicht rechtzeitig
  LLMResponseError           HTTP-Fehler, kein JSON, fehlendes 'response'-Feld
```

`LLMConnectionError` und `LLMTimeoutError` sind transient und werden bis zu
`max_retries` mal wiederholt. `LLMResponseError` ist deterministisch. Ein Retry
würde dasselbe Ergebnis liefern und wird deshalb nicht wiederholt.

In [ ]:
# Verbindungstest: roher HTTP-Aufruf ohne Strategie oder Template
client = OllamaClient(base_config)

OLLAMA_AVAILABLE = False

try:
    t0  = time.perf_counter()
    raw = client.generate("In einem Satz: Was ist ein Sprachmodell?")
    latency_ms = (time.perf_counter() - t0) * 1000

    OLLAMA_AVAILABLE = True
    print(f"Ollama verfügbar. Latenz: {latency_ms:.0f} ms")
    print(f"Modell: {raw.get('model', 'unbekannt')}")
    print(f"Antwort: {raw['response'].strip()[:120]}")

except LLMConnectionError as e:
    print(f"Verbindungsfehler: {e.message}")
    print("Prüfen: läuft 'ollama serve' und ist Port 11434 erreichbar?")
except LLMTimeoutError as e:
    print(f"Timeout: {e.message}")
    print("Bei CPU-Inferenz: timeout in GenerationConfig erhöhen.")
except LLMResponseError as e:
    print(f"Response-Fehler: {e.message}")

In [ ]:
# Token-Statistiken aus dem Ollama-Response
if OLLAMA_AVAILABLE and 'raw' in dir():
    eval_count    = raw.get("eval_count")
    prompt_eval   = raw.get("prompt_eval_count")
    eval_duration = raw.get("eval_duration")

    if eval_count and eval_duration:
        tps = eval_count / (eval_duration / 1e9)
        print(f"Prompt-Token       : {prompt_eval}")
        print(f"Generierte Token   : {eval_count}")
        print(f"Inferenz-Tempo     : {tps:.1f} Token/s")
        print()
        print("Typisch für CPU-Inferenz (7B-Modell): 3-15 Token/s.")
    else:
        print("Keine Token-Statistiken im Response (Ollama-Versions-abhängig).")
else:
    print("Kein Ollama-Response verfügbar, Verbindungstest fehlgeschlagen.")

## 7. ContextPreparer: Zeichen-Budget statt Tokenizer

`ContextPreparer` transformiert rohen Retrieval-Output in bereinigte, deduplizierte und
budget-begrenzte Text-Chunks.

### Warum kein Tokenizer?

Die naheliegende Alternative wäre, einen Tokenizer zu laden, die Chunks zu tokenisieren und
Token zu zählen. So machen es Cloud-APIs intern. Dieses System verwendet stattdessen ein
Zeichen-Budget, mit klaren Vor- und Nachteilen:

**Vorteile:**
- keine zusätzliche Abhängigkeit (`tiktoken`, `transformers`)
- funktioniert für jedes Modell ohne modellspezifische Konfiguration
- deterministisch und schnell

**Nachteile:**
- ungenau, weil das Verhältnis Zeichen zu Token schwankt (Englisch etwa 4, Deutsch etwa 5 bis
  6, Code 1 bis 2 Zeichen pro Token)
- bei token-kritischen Prompts kann man zu viel oder zu wenig abschneiden
- `max_context_chars=4000` entspricht grob 700 bis 1400 Token, also mit Unsicherheit behaftet

Für Lehre und Reproduzierbarkeit ist die Vereinfachung bewusst gewählt: Sie kommt ohne
schwere Abhängigkeit aus und ist vorhersagbar. In produktiven Systemen mit definierten SLAs
würde man einen modellspezifischen Tokenizer einbinden.

**Zum Experimentieren:** `max_context_chars` ist der zentrale Hebel. Die nächsten Zellen zeigen
denselben Retrieval-Output einmal mit großem und einmal mit sehr kleinem Budget. Beobachte,
welche Chunks bei knappem Budget zuerst herausfallen. Weil der Retriever nach Relevanz
sortiert hat, überleben die relevantesten Chunks, die am wenigsten relevanten fallen zuerst
weg.


In [ ]:
# ContextPreparer mit den echten Retrieval-Ergebnissen
preparer = ContextPreparer(max_context_chars=base_config.max_context_chars)

# ContextPreparer.prepare() extrahiert den 'text'-Wert automatisch.
bereinigte_chunks = preparer.prepare(retrieval_results)

print(f"Eingabe (Retrieval-Ergebnisse) : {len(retrieval_results)} Items")
print(f"Ausgabe (bereinigte Chunks)    : {len(bereinigte_chunks)} Strings")
print(f"Gesamt-Zeichen                 : {sum(len(c) for c in bereinigte_chunks)}")
print(f"Budget                         : {base_config.max_context_chars}")
print()

fallen_chunks = len(retrieval_results) - len(bereinigte_chunks)
if fallen_chunks > 0:
    print(f"Budget-Abschnitt: {fallen_chunks} Chunk(s) wurden wegen des Zeichen-Budgets verworfen.")
    print()
    print("Wichtig: Der Retriever hat nach Relevanz geordnet. Was zuerst kommt,")
    print("hat die höchste Relevanz und landet sicher im Kontext.")
    print("Was zuletzt kommt, hat die niedrigste Relevanz und fällt zuerst heraus.")
else:
    print("Alle Retrieval-Ergebnisse passen ins Budget.")

print()
for i, chunk in enumerate(bereinigte_chunks):
    print(f"  [{i}] {len(chunk):>5} Zeichen  {chunk[:70].replace(chr(10), ' ')!r}...")

In [ ]:
# Budget-Verhalten mit kleinem Budget demonstrieren
kleines_budget = ContextPreparer(max_context_chars=200)

ergebnis = kleines_budget.prepare(retrieval_results)

print(f"Budget: 200 Zeichen")
print(f"Eingabe: {len(retrieval_results)} Retrieval-Ergebnisse")
print(f"Ausgabe: {len(ergebnis)} Chunks ({sum(len(c) for c in ergebnis)} Zeichen)")
print()
print("Erhaltene Chunks:")
for chunk in ergebnis:
    print(f"  '{chunk[:80].replace(chr(10), ' ')}'...")
    print(f"  ({len(chunk)} Zeichen)")
print()
print("Die Implikation: ein zu kleines Budget verwirft wichtige Chunks.")
print("Ein zu großes Budget überschreitet das Kontextfenster des Modells.")
print("Das optimale Budget liegt zwischen diesen Extremen, und ist modellabhängig.")

### Die versteckte Implikation des Budget-Verhaltens

Das Budget-Verhalten hat eine direkte Konsequenz für das Retrieval-Design: Was der Retriever
als Erstes zurückgibt, landet mit der höchsten Wahrscheinlichkeit im Kontext. Retrieval-Ranking
ist deshalb keine kosmetische Detail-Entscheidung, sondern bestimmt direkt, welche
Informationen das Sprachmodell überhaupt zu sehen bekommt.

Schlechtes Retrieval-Ranking mit einem guten Sprachmodell führt oft zu einer schlechten
Antwort. Gutes Retrieval-Ranking mit einem mittelmäßigen Sprachmodell führt oft zu einer
akzeptablen Antwort. In der Praxis ist Retrieval-Tuning daher häufig wirkungsvoller als ein
Modellwechsel.


## 8. PromptBuilder und Templates: Prompt-Engineering als kontrollierte Variable

### Warum Templates als Objekte?

Die naive Alternative wäre, Prompt-Strings direkt im Code zusammenzusetzen. In Experimenten
entstehen daraus drei Probleme:

- unterschiedliche Prompt-Varianten sind schwer zu verfolgen
- Änderungen sind nicht versioniert
- `template_name` und `template_version` können nicht im Ergebnis gespeichert werden

`PromptTemplate` ist ein eingefrorenes Dataclass mit Name und Version. Jedes
`GenerationResult` trägt `template_name` und `template_version`, was die Zuordnung in einer
Ergebnis-Tabelle ermöglicht.

**Zum Experimentieren:** Die mitgelieferten Templates (`STRICT_RAG_TEMPLATE`, `COT_TEMPLATE`,
`FEW_SHOT_TEMPLATE`) verfolgen unterschiedliche Strategien, von strikter Kontextbindung über
schrittweises Schließen bis zu Beispiel-gestütztem Antworten. Ein eigenes Template ist schnell
gebaut (siehe Abschnitt 12), Pflicht ist nur, dass das `user_template` ein `{context}`-Feld
enthält. In Produktivsystemen werden Templates versioniert und mit Experiment-IDs verknüpft,
damit jedes Ergebnis einem konkreten Prompt zugeordnet bleibt.


In [ ]:
# Templates inspizieren: was genau wird an das Modell geschickt?
def zeige_template_kurz(template):
    print(f"Name     : {template.name} v{template.version}")
    print(f"Separator: {repr(template.separator)}")
    print("System-Prompt:")
    for line in template.system_prompt.split("\n"):
        print(f"  {line}")
    print()

for t in [STRICT_RAG_TEMPLATE, COT_TEMPLATE, FEW_SHOT_TEMPLATE]:
    zeige_template_kurz(t)
    print("-" * 60)

In [ ]:
# Finalen Prompt aus echten retrieved Chunks zusammensetzen
builder = PromptBuilder(STRICT_RAG_TEMPLATE)

finaler_prompt = builder.build(
    query=erste_query,
    context_chunks=bereinigte_chunks,
)

print(f"Query            : {erste_query!r}")
print(f"Kontext-Chunks   : {len(bereinigte_chunks)}")
print(f"Kontext-Zeichen  : {sum(len(c) for c in bereinigte_chunks)}")
print(f"Prompt-Zeichen   : {len(finaler_prompt)}")
print()
print("Vollständiger Prompt (genau so an Ollama gesendet):")
print("-" * 70)
print(finaler_prompt[:1200])
if len(finaler_prompt) > 1200:
    print(f"... [{len(finaler_prompt) - 1200} weitere Zeichen]")
print("-" * 70)

## 9. SimpleRAGStrategy: vollständiger Aufruf mit echten Retrieval-Daten

`SimpleRAGStrategy` führt alle Schritte in einer Sequenz aus:

1. `ContextPreparer.prepare(retrieval_results)` liefert die bereinigten Chunks
2. `PromptBuilder.build(query, chunks)` baut den finalen Prompt
3. `OllamaClient.generate(prompt)` ruft das Modell auf
4. `GenerationResult(...)` bündelt Antwort und Messwerte

Ein einziger LLM-Aufruf, linearer Ablauf, kein Zustand zwischen den Aufrufen.

### Dependency Injection

Die Strategie bekommt `OllamaClient`, `PromptBuilder` und `ContextPreparer` von außen
übergeben, sie erstellt sie nicht selbst. Das erlaubt im Notebook, Client, Builder und
Preparer als getrennte Variablen mit verschiedenen Konfigurationen zu halten und frei zu
kombinieren. Genau das macht die einzelnen Experimente in den folgenden Abschnitten möglich.


In [ ]:
if not OLLAMA_AVAILABLE:
    print("Ollama nicht verfügbar. Abschnitte mit LLM-Aufrufen werden übersprungen.")
else:
    simple_config = GenerationConfig(
        model_name="mistral",
        temperature=0.0,
        max_tokens=400,
        max_context_chars=base_config.max_context_chars,
    )

    simple_strategy = SimpleRAGStrategy(
        client=OllamaClient(simple_config),
        prompt_builder=PromptBuilder(STRICT_RAG_TEMPLATE),
        context_preparer=ContextPreparer(max_context_chars=simple_config.max_context_chars),
    )

    # Echte Query + echte Retrieval-Ergebnisse
    simple_result = simple_strategy.generate(erste_query, retrieval_results)

    print(format_result_summary(simple_result))
    print()

    if simple_result.success:
        print("Vollständige Antwort:")
        print("-" * 70)
        print(simple_result.answer.strip())
        print("-" * 70)
    else:
        print(f"Fehler: {simple_result.error}")

In [ ]:
if OLLAMA_AVAILABLE and 'simple_result' in dir() and simple_result.success:
    print("GenerationResult, alle Felder:")
    print()
    for key, val in simple_result.to_dict().items():
        if key == "prompt":
            print(f"  {key:<22}: [{len(val)} Zeichen]")
        elif isinstance(val, str) and len(val) > 80:
            print(f"  {key:<22}: [{len(val)} Zeichen]")
        elif isinstance(val, dict) and "response" in str(key).lower():
            print(f"  {key:<22}: [dict, {len(val)} Keys]")
        else:
            print(f"  {key:<22}: {val}")

---

## 10. Kontextqualität als primärer Qualitätstreiber

### Die zentrale These

In RAG-Systemen ist Kontextqualität häufig wichtiger als Modellgröße.
Ein schlechter Retriever liefert irrelevante Chunks an ein ausgezeichnetes Modell,
und das Modell kann daraus keine gute Antwort konstruieren, wenn es seine Instruktionen einhält.

Das ist gleichzeitig die häufigste Ursache für schlechte RAG-Systemleistung in der Praxis:
nicht das Modell, sondern das Retrieval.

Dieser Abschnitt demonstriert das systematisch mit drei Szenarien.

In [ ]:
if not OLLAMA_AVAILABLE:
    print("Ollama nicht verfügbar. Kontextqualitäts-Experimente übersprungen.")
else:
    kontext_config = GenerationConfig(model_name="mistral", temperature=0.0, max_tokens=300)
    kontext_preparer = ContextPreparer(max_context_chars=kontext_config.max_context_chars)
    kontext_builder  = PromptBuilder(STRICT_RAG_TEMPLATE)

    kontext_query = "Was ist der Unterschied zwischen HNSW und einem Flat Index?"

    # --- Szenario A: kein Kontext ---
    r_kein = SimpleRAGStrategy(
        OllamaClient(kontext_config), kontext_builder, kontext_preparer
    ).generate(kontext_query, [])

    # --- Szenario B: irrelevanter Kontext ---
    # Diese Chunks kommen aus dem echten Index, aber für eine andere Query.
    irrelevant_query = "Geschichte der Eiffelturm-Konstruktion"
    irrelevante_results = retrieval_orch.retrieve_with_trace(irrelevant_query, k=3)["final_results"]

    r_irrelevant = SimpleRAGStrategy(
        OllamaClient(kontext_config), kontext_builder, kontext_preparer
    ).generate(kontext_query, irrelevante_results)

    # --- Szenario C: relevanter Kontext (manuell, falls real nicht verfügbar) ---
    relevanter_kontext = [
        {
            "text": (
                "A flat index performs exact nearest-neighbor search by comparing the query "
                "vector to every stored vector. It guarantees exact results but scales as O(n) "
                "with the number of vectors, making it impractical for large collections."
            ),
            "score": 0.92, "chunk_id": "manual-1", "document_id": "manual",
        },
        {
            "text": (
                "HNSW (Hierarchical Navigable Small World) is a graph-based approximate "
                "nearest-neighbor algorithm. At query time, it navigates the graph "
                "hierarchically, achieving sub-linear search complexity at the cost of "
                "approximation. The ef_construction and M parameters control the "
                "recall/speed trade-off."
            ),
            "score": 0.89, "chunk_id": "manual-2", "document_id": "manual",
        },
    ]

    r_relevant = SimpleRAGStrategy(
        OllamaClient(kontext_config), kontext_builder, kontext_preparer
    ).generate(kontext_query, relevanter_kontext)

    print(f"Query: {kontext_query!r}")
    print()

    for label, result in [
        ("Szenario A: Kein Kontext",          r_kein),
        ("Szenario B: Irrelevanter Kontext",   r_irrelevant),
        ("Szenario C: Relevanter Kontext",     r_relevant),
    ]:
        antwort = result.answer.strip() if result.answer.strip() else "[leere Antwort]"
        print(f"--- {label} ---")
        print(f"Kontext-Zeichen: {result.context_chars}")
        print(antwort[:300])
        print()

### Was Halluzination im RAG-Kontext bedeutet

Sprachmodelle haben Weltwissen aus dem Training. Wenn das Template keine strikte
Kontextbindung erzwingt, oder wenn das Modell die Instruktion überstimmt, kann es
Antworten aus dem Trainingsgedächtnis generieren, auch wenn kein passender Kontext
vorhanden ist.

`STRICT_RAG_TEMPLATE` instruiert das Modell, mit "I don't know." zu antworten,
wenn der Kontext nicht ausreicht. Das ist keine Garantie. Es ist eine Instruktion.
Kleinere Modelle halten sich weniger zuverlässig daran als größere.

Szenario B ist besonders tückisch: das Modell bekommt Kontext, der real ist,
aber zur Frage irrelevant. In einem echten System wäre das schwer zu erkennen,
weil der Retrieval-Score oberflächlich plausibel aussieht.

In [ ]:
if OLLAMA_AVAILABLE:
    # Halluzinations-Test: interne Information, die das Modell nicht kennen kann
    halluz_config = GenerationConfig(model_name="mistral", temperature=0.0, max_tokens=150)

    interne_frage = (
        "What did the internal engineering review conclude about the HNSW index "
        "configuration used in the Acme Corp knowledge base on 2024-03-15?"
    )

    r_halluz = SimpleRAGStrategy(
        OllamaClient(halluz_config),
        PromptBuilder(STRICT_RAG_TEMPLATE),
        ContextPreparer(max_context_chars=halluz_config.max_context_chars),
    ).generate(interne_frage, [
        {"text": "HNSW is a graph-based approximate nearest-neighbor algorithm.", "score": 0.7}
    ])

    print("Halluzinations-Test: Frage nach unmöglich bekannter interner Information")
    print()
    print("Antwort:")
    print(r_halluz.answer.strip())
    print()
    print("Erwartetes Verhalten bei strikter Instruktionsbefolgung: 'I don't know.'")
    print("Wenn das Modell eine Antwort erfindet, zeigt das die Grenze der")
    print("Instruktionsfolge-Fähigkeit dieses Modells.")
else:
    print("Ollama nicht verfügbar, Halluzinationstest übersprungen.")

---

## 11. Retrieval-Fails und ihre direkte Wirkung auf Generation

Dieser Abschnitt zeigt, was passiert, wenn Retrieval schlechte Ergebnisse liefert,
mit echten Retrieval-Aufrufen aus dem Index.

Das Ziel ist zu verstehen, warum es keine "Generation-seitige Lösung" für
schlechtes Retrieval gibt: das Modell bekommt, was es bekommt.

In [ ]:
if OLLAMA_AVAILABLE:
    test_queries_retrieval = [
        # (Query, erwartete Qualität des Retrievals)
        ("Wie funktioniert Embedding in einem RAG-System?",   "gut, direkt im Corpus"),
        ("Welche Farbe hat das Büro von Anthropic in London?", "schlecht, nicht im Corpus"),
        ("sparse retrieval bm25",                              "mittel, technische Terme"),
    ]

    retrieval_gen_config = GenerationConfig(
        model_name="mistral", temperature=0.0, max_tokens=200
    )

    print("Retrieval-Qualität und ihr Einfluss auf Generation:")
    print()

    for query, erwartung in test_queries_retrieval:
        r_results = retrieval_orch.retrieve_with_trace(query=query, k=3)["final_results"]
        top_score = r_results[0]["score"] if r_results else 0.0

        gen_result = SimpleRAGStrategy(
            OllamaClient(retrieval_gen_config),
            PromptBuilder(STRICT_RAG_TEMPLATE),
            ContextPreparer(max_context_chars=retrieval_gen_config.max_context_chars),
        ).generate(query, r_results)

        print(f"Query    : {query!r}")
        print(f"Erwartung: {erwartung}")
        print(f"Top Score: {top_score:.5f}  ({len(r_results)} Chunks)")
        print(f"Antwort  : {gen_result.answer.strip()[:200]}")
        print()
else:
    print("Ollama nicht verfügbar, Retrieval-Fail-Test übersprungen.")

---

## 12. Template-Vergleich mit echten Retrieval-Daten

Das Modell, der Kontext und die Frage bleiben konstant. Nur das Template ändert sich.
Das zeigt, welchen Einfluss das Template auf Antwort-Struktur, Länge und
Reasoning-Tiefe hat, und zwar mit denselben echten abgerufenen Chunks.

In [ ]:
# Eigenes Template für kurze, präzise Antworten
BULLET_TEMPLATE = PromptTemplate(
    name="bullet_strict",
    version="1.0.0",
    system_prompt=(
        "You are a technical assistant. Answer only from the provided context.\n"
        "Format your answer as 2-4 concise bullet points.\n"
        "If the context is insufficient, write: 'Insufficient context.'\n"
        "Do not add information from your training data."
    ),
    user_template=(
        "Context:\n"
        "{context}\n\n"
        "Question:\n"
        "{query}\n\n"
        "Bullet points:"
    ),
)
print(f"Eigenes Template erstellt: {BULLET_TEMPLATE.name} v{BULLET_TEMPLATE.version}")
print()

# Validierung: fehlendes Pflichtfeld
try:
    PromptTemplate(
        name="broken", version="1.0.0",
        system_prompt="Du bist ein Assistent.",
        user_template="Beantworte: {query}"  # {context} fehlt
    )
except ValueError as e:
    print(f"Validierungsfehler korrekt: {e}")

In [ ]:
if OLLAMA_AVAILABLE:
    template_config = GenerationConfig(
        model_name="mistral", temperature=0.0, max_tokens=400
    )
    template_preparer = ContextPreparer(max_context_chars=template_config.max_context_chars)

    template_query = "Wie beeinflusst die Chunking-Strategie die Qualität des Retrievals?"
    template_results = retrieval_orch.retrieve_with_trace(template_query, k=4)["final_results"]

    templates_zum_vergleich = [
        ("STRICT_RAG",       STRICT_RAG_TEMPLATE),
        ("CHAIN_OF_THOUGHT",  COT_TEMPLATE),
        ("FEW_SHOT",          FEW_SHOT_TEMPLATE),
        ("BULLET_STRICT",     BULLET_TEMPLATE),
    ]

    template_ergebnisse = []

    for label, tmpl in templates_zum_vergleich:
        strategy = SimpleRAGStrategy(
            OllamaClient(template_config),
            PromptBuilder(tmpl),
            template_preparer,
        )
        result = strategy.generate(template_query, template_results)
        template_ergebnisse.append((label, result))

        print(f"Template: {label} ({result.latency_ms:.0f} ms | {result.prompt_chars} Prompt-Zeichen)")
        print("-" * 60)
        print(result.answer.strip()[:300])
        print()

    print()
    print(f"{'Template':<20} {'Prompt-Zeichen':>15} {'Antwort-Zeichen':>16} {'Latenz (ms)':>12}")
    print("-" * 65)
    for label, result in template_ergebnisse:
        print(f"{label:<20} {result.prompt_chars:>15} {len(result.answer):>16} {result.latency_ms:>12.0f}")
else:
    print("Ollama nicht verfügbar, Template-Vergleich übersprungen.")

## 13. Temperature: Determinismus, Sampling und was das für RAG bedeutet

Sprachmodelle erzeugen für jedes Token eine Wahrscheinlichkeitsverteilung über das Vokabular.
`temperature` skaliert diese Verteilung:

- `temperature=0.0`: greedy decoding, immer das wahrscheinlichste Token. Bei identischem Seed
  und Modell entsteht dieselbe Ausgabe.
- `temperature=1.0`: die originale Verteilung, Sampling nach den Trainingswahrscheinlichkeiten.
- `temperature>1.0`: die Verteilung wird flacher, der Output wird kreativer, aber auch
  inkohärenter.

### Relevanz für RAG-Systeme

Bei faktenbezogenen Frage-Antwort-Systemen ist Determinismus meistens erwünscht. Gleiche Frage
und gleicher Kontext ergeben dann dieselbe Antwort, was reproduzierbar und testbar ist. Eine
höhere Temperature erhöht dagegen das Risiko, dass das Modell über den Kontext hinausgeht.

**Zum Experimentieren:** Die nächste Zelle ruft dieselbe Query mit `temperature` 0.0, 0.5 und
1.2 auf. Achte darauf, ob und ab wann die Antworten ungenauer werden oder den Kontext verlassen.
In Produktivsystemen wählt man für faktische Antworten meist eine niedrige Temperature und hebt
sie nur dort an, wo bewusst sprachliche Vielfalt gewünscht ist.


In [ ]:
if OLLAMA_AVAILABLE:
    temp_query = "What are the key trade-offs when choosing between dense and sparse retrieval?"
    temp_retrieval = retrieval_orch.retrieve_with_trace(temp_query, k=3)["final_results"]
    temp_preparer  = ContextPreparer(max_context_chars=3000)

    temperaturen = [0.0, 0.5, 1.2]
    ergebnisse_temp = []

    for temp in temperaturen:
        cfg = GenerationConfig(model_name="mistral", temperature=temp, max_tokens=200, seed=42)
        strategy = SimpleRAGStrategy(
            OllamaClient(cfg),
            PromptBuilder(STRICT_RAG_TEMPLATE),
            temp_preparer,
        )
        result = strategy.generate(temp_query, temp_retrieval)
        ergebnisse_temp.append((temp, result))
        print(f"temperature={temp:.1f} | latenz={result.latency_ms:.0f} ms")
        print(result.answer.strip()[:200])
        print()
else:
    print("Ollama nicht verfügbar, Temperature-Experiment übersprungen.")

In [ ]:
if OLLAMA_AVAILABLE:
    # Reproduzierbarkeits-Test: temperature=0.0, zwei identische Aufrufe
    cfg_det = GenerationConfig(model_name="mistral", temperature=0.0, max_tokens=100, seed=42)
    det_strategy = SimpleRAGStrategy(
        OllamaClient(cfg_det),
        PromptBuilder(STRICT_RAG_TEMPLATE),
        ContextPreparer(max_context_chars=cfg_det.max_context_chars),
    )

    repro_query = "Explain the purpose of ContextPreparer in one sentence."
    repro_chunks = bereinigte_chunks[:2] if bereinigte_chunks else []

    r1 = det_strategy.generate(repro_query, repro_chunks)
    r2 = det_strategy.generate(repro_query, repro_chunks)

    identisch = r1.answer.strip() == r2.answer.strip()
    print(f"Antworten identisch bei temperature=0.0: {identisch}")

    if not identisch:
        print()
        print("Antwort 1:", r1.answer.strip()[:150])
        print("Antwort 2:", r2.answer.strip()[:150])
        print()
        print("Nicht-Determinismus trotz temperature=0.0 kann entstehen durch:")
        print("- NUMA-Architekturen mit parallelen Operationen")
        print("- Nicht-deterministisches Float-Rounding bei CPU-parallelen Ops")
        print("'seed' allein garantiert auf CPU-Ebene keine absolute Reproduzierbarkeit.")
else:
    print("Ollama nicht verfügbar, Reproduzierbarkeits-Test übersprungen.")

## 14. RefineRAGStrategy: iterative Verfeinerung

### Das Algorithmus-Design

RefineRAG verarbeitet die Chunks nacheinander und verbessert die Antwort Schritt für Schritt:

1. Chunk 0 erzeugt mit dem Initial-Template eine erste Antwort
2. Chunk 1 verfeinert diese Antwort mit dem Update-Template
3. Chunk 2 verfeinert das Ergebnis erneut
4. und so weiter, bis der letzte Chunk die finale Antwort liefert

Die Anzahl der LLM-Aufrufe entspricht der Anzahl der Chunks. Bei 5 Chunks sind das 5 Aufrufe.

### Wann RefineRAG sinnvoll ist und wann nicht

**Sinnvoll,** wenn die relevante Information auf viele Chunks verteilt ist, ein einzelner
Prompt nicht alles gleichzeitig integrieren kann und Antwortqualität wichtiger ist als Latenz.

**Nicht sinnvoll,** wenn die Information problemlos in einen Prompt passt, Latenz kritisch ist
oder viele Chunks wenig relevanten Inhalt haben, sodass jeder Verfeinerungsschritt nur Zeit
kostet.

### Skalierungsproblem

Bei 20 Chunks und 2000 ms pro LLM-Aufruf ergibt das 40 Sekunden Gesamtlatenz. Für interaktive
Nutzung ist das zu viel. Übliche Gegenmittel sind ein Reranker (nur die Top-3 statt Top-20
verfeinern) oder eine MapReduce-artige Parallelisierung.


In [ ]:
if OLLAMA_AVAILABLE:
    refine_config   = GenerationConfig(model_name="mistral", temperature=0.0, max_tokens=400)
    refine_preparer = ContextPreparer(max_context_chars=refine_config.max_context_chars)

    refine_query = "Describe the complete architecture of a local RAG system."
    refine_retrieval = retrieval_orch.retrieve_with_trace(refine_query, k=4)["final_results"]

    refine_strategy = RefineRAGStrategy(
        client=OllamaClient(refine_config),
        context_preparer=refine_preparer,
    )

    print(f"Query  : {refine_query!r}")
    print(f"Chunks : {len(refine_retrieval)}")
    print(f"Erwartet: {len(refine_retrieval)} LLM-Aufrufe")
    print()

    refine_result = refine_strategy.generate(refine_query, refine_retrieval)

    print(format_result_summary(refine_result))
    print()
    print("Finale Antwort nach iterativer Verfeinerung:")
    print(refine_result.answer.strip())
else:
    print("Ollama nicht verfügbar, RefineRAG übersprungen.")

---

## 15. SimpleRAG vs. RefineRAG: kontrollierter Vergleich

Identische Query, identische Retrieval-Ergebnisse, identisches Modell.
Nur die Strategie ändert sich. Der Latenz-Unterschied ist direkt messbar.

In [ ]:
if OLLAMA_AVAILABLE:
    compare_config   = GenerationConfig(model_name="mistral", temperature=0.0, max_tokens=350)
    compare_preparer = ContextPreparer(max_context_chars=compare_config.max_context_chars)

    compare_query    = "How do dense and sparse retrieval complement each other?"
    compare_results  = retrieval_orch.retrieve_with_trace(compare_query, k=3)["final_results"]

    simple_cmp = SimpleRAGStrategy(
        OllamaClient(compare_config),
        PromptBuilder(STRICT_RAG_TEMPLATE),
        compare_preparer,
    )
    refine_cmp = RefineRAGStrategy(
        OllamaClient(compare_config),
        compare_preparer,
    )

    r_simple = simple_cmp.generate(compare_query, compare_results)
    r_refine = refine_cmp.generate(compare_query, compare_results)

    n_chunks = len(ContextPreparer(max_context_chars=compare_config.max_context_chars).prepare(compare_results))

    print(f"{'Strategie':<22} {'LLM-Aufrufe':>12} {'Latenz (ms)':>12} {'Antwort-Zeichen':>16}")
    print("-" * 64)
    print(f"{'SimpleRAGStrategy':<22} {1:>12} {r_simple.latency_ms:>12.0f} {len(r_simple.answer):>16}")
    print(f"{'RefineRAGStrategy':<22} {n_chunks:>12} {r_refine.latency_ms:>12.0f} {len(r_refine.answer):>16}")

    if r_refine.latency_ms > 0 and r_simple.latency_ms > 0:
        faktor = r_refine.latency_ms / r_simple.latency_ms
        print(f"\nLatenz-Faktor RefineRAG / SimpleRAG: {faktor:.1f}x")
        print(f"(Bei {n_chunks} Chunks erwartet: ca. {n_chunks}x)")

    print()
    print("Simple-Antwort:")
    print(r_simple.answer.strip()[:300])
    print()
    print("Refine-Antwort (nach iterativer Verfeinerung):")
    print(r_refine.answer.strip()[:300])
else:
    print("Ollama nicht verfügbar, Strategie-Vergleich übersprungen.")

## 16. Vollständiger Pipeline-Durchlauf

Dieser Abschnitt führt die vollständige Pipeline in einem Block aus: echtes Retrieval, dann
`ContextPreparer`, `PromptBuilder`, der Aufruf an Ollama und schließlich das `GenerationResult`.

Der Code zeigt, wie alle Komponenten zusammenarbeiten und welche Variablen sich an welchen
Stellen austauschen lassen.


In [ ]:
if OLLAMA_AVAILABLE:
    # --- Konfiguration ---
    pipeline_config = GenerationConfig(
        model_name="mistral",
        temperature=0.0,
        max_tokens=512,
        max_context_chars=3500,
    )

    # --- Query ---
    nutzer_query = "What are the advantages and limitations of RAG compared to pure language models?"

    # --- Retrieval (echt, mit Trace) ---
    pipeline_trace    = retrieval_orch.retrieve_with_trace(query=nutzer_query, k=5)
    pipeline_retrieval = pipeline_trace["final_results"]

    print(f"Retrieval abgeschlossen:")
    print(f"  dense_candidates  : {len(pipeline_trace['dense_candidates'])}")
    print(f"  sparse_candidates : {len(pipeline_trace['sparse_candidates'])}")
    print(f"  fused_candidates  : {len(pipeline_trace['fused_candidates'])}")
    print(f"  final_results     : {len(pipeline_retrieval)}")
    print()

    # --- Pipeline aufbauen ---
    pipeline_client   = OllamaClient(pipeline_config)
    pipeline_preparer = ContextPreparer(max_context_chars=pipeline_config.max_context_chars)
    pipeline_builder  = PromptBuilder(COT_TEMPLATE)  # Chain-of-Thought
    pipeline_strategy = SimpleRAGStrategy(pipeline_client, pipeline_builder, pipeline_preparer)

    # --- Ausführen ---
    pipeline_result = pipeline_strategy.generate(nutzer_query, pipeline_retrieval)

    # --- Ergebnis ---
    print(f"Query             : {nutzer_query!r}")
    print(f"Kontext-Zeichen   : {pipeline_result.context_chars}")
    print(f"Prompt-Zeichen    : {pipeline_result.prompt_chars}")
    print(f"Latenz            : {pipeline_result.latency_ms:.0f} ms")
    print(f"Template          : {pipeline_result.template_name} v{pipeline_result.template_version}")
    print(f"Erfolgreich       : {pipeline_result.success}")
    print()
    print("Antwort:")
    print("-" * 70)
    print(pipeline_result.answer.strip())
else:
    print("Ollama nicht verfügbar, vollständiger Pipeline-Durchlauf übersprungen.")

---

## 17. Systemgrenzen: was diese Implementierung bewusst nicht kann

Das Folgende ist keine Liste von Fehlern. Es ist eine Liste bewusster Design-Entscheidungen.
Zu verstehen, was fehlt und warum, ist genauso wichtig wie zu verstehen, was vorhanden ist.

### Kein Streaming

`OllamaClient` verwendet `stream=False`. Das Modell generiert den gesamten Text,
bevor die Antwort zurückgegeben wird. In produktiven Web-Anwendungen würde man
Streaming verwenden. Der Nutzer sieht Token, während sie generiert werden.
Das verbessert die wahrgenommene Latenz erheblich.

Für Notebooks ist Streaming unnötig komplex: Token-by-Token-Ausgabe in einer
Notebook-Cell erfordert `ipywidgets` oder Ähnliches. Die Gesamtlatenz bleibt dieselbe.


### Kein Re-Ranker

Die Retrieval-Ausgabe wird in der gelieferten Reihenfolge verarbeitet. Ein Re-Ranker
würde initiale Retrieval-Ergebnisse mit einem zweiten Modell neu bewerten, bevor sie
an den Generator gehen. Das verbessert die Kontext-Qualität, fügt aber einen weiteren
Inferenzschritt hinzu.

### Kein Tokenizer

Zeichen-Budget statt Token-Budget. Das ist eine Approximation, wie in Abschnitt 7
dokumentiert.

### Keine Quellen-Attribution

`GenerationResult` speichert die verwendeten Chunks nicht als separate Felder mit
Quell-Metadaten. In einem produktiven System würde man die `document_id`- und
`chunk_id`-Felder der Retrieval-Ergebnisse durch die Pipeline weiterführen und
im Ergebnis ausgeben.

In [ ]:
if OLLAMA_AVAILABLE:
    # Latenz-Skalierung: RefineRAG mit wachsender Chunk-Anzahl
    skalierungs_config   = GenerationConfig(model_name="mistral", temperature=0.0, max_tokens=150)
    skalierungs_preparer = ContextPreparer(max_context_chars=skalierungs_config.max_context_chars)

    skalierungs_queries = [
        "What is vector search?",
        "Explain BM25 ranking in retrieval systems.",
        "How does context preparation work in RAG?",
    ]

    skalierungs_results = []
    for q in skalierungs_queries:
        r = retrieval_orch.retrieve_with_trace(q, k=2)["final_results"]
        skalierungs_results.extend(r)

    # Chunks de-duplizieren nach ID
    seen_ids = set()
    unique_results = []
    for r in skalierungs_results:
        if r["id"] not in seen_ids:
            seen_ids.add(r["id"])
            unique_results.append(r)

    print(f"Skalierungstest mit bis zu {len(unique_results)} einzigartigen Chunks.")
    print()
    print(f"{'Chunks':>7} {'Latenz (ms)':>12} {'ms/Chunk':>10}")
    print("-" * 35)

    for n in range(1, min(len(unique_results) + 1, 5)):
        test_chunks = unique_results[:n]
        strat = RefineRAGStrategy(OllamaClient(skalierungs_config), skalierungs_preparer)
        r = strat.generate("What is RAG?", test_chunks)
        ms_per_chunk = r.latency_ms / n if n > 0 else 0
        print(f"  {n:>5}  {r.latency_ms:>12.0f}  {ms_per_chunk:>10.0f}")

    print()
    if len(unique_results) >= 2:
        strat_base = RefineRAGStrategy(OllamaClient(skalierungs_config), skalierungs_preparer)
        r_base = strat_base.generate("What is RAG?", unique_results[:2])
        ms_per = r_base.latency_ms / 2
        print(f"Hochrechnung: Bei 20 Chunks würde RefineRAG ca. {int(ms_per * 20 / 1000)} Sekunden benötigen.")
        print("Das ist die praktische Obergrenze für interaktive Nutzung.")
else:
    print("Ollama nicht verfügbar, Skalierungstest übersprungen.")

---

## 18. Freie Experimente: strukturierte Parametervariationen

Der folgende Block ist so aufgebaut, dass alle relevanten Parameter an einer Stelle
stehen und die Ausgabe alle Metriken zeigt, die für eine strukturierte Analyse nützlich sind.

Sinnvolle Variationen:
- Template wechseln und Ausgabestruktur vergleichen
- `max_context_chars` reduzieren und beobachten, welche Chunks wegfallen
- Eigene Query gegen den echten Index stellen
- `strategy = "refine"` statt `"simple"`, Latenz-Unterschied messen

In [ ]:
if OLLAMA_AVAILABLE:
    # ---------- Parameter ----------
    EXPERIMENT_QUERY       = "How does the context character budget affect retrieval and generation?"
    EXPERIMENT_TEMPLATE    = STRICT_RAG_TEMPLATE   # COT_TEMPLATE, FEW_SHOT_TEMPLATE, BULLET_TEMPLATE
    EXPERIMENT_STRATEGY    = "simple"               # "simple" oder "refine"
    EXPERIMENT_TEMPERATURE = 0.0                   # [0.0, 2.0]
    EXPERIMENT_MAX_TOKENS  = 400
    EXPERIMENT_MAX_CONTEXT = 3000
    EXPERIMENT_TOP_K       = 4
    # --------------------------------

    exp_retrieval = retrieval_orch.retrieve_with_trace(EXPERIMENT_QUERY, k=EXPERIMENT_TOP_K)["final_results"]

    exp_config   = GenerationConfig(
        model_name="mistral",
        temperature=EXPERIMENT_TEMPERATURE,
        max_tokens=EXPERIMENT_MAX_TOKENS,
        max_context_chars=EXPERIMENT_MAX_CONTEXT,
    )
    exp_preparer = ContextPreparer(max_context_chars=exp_config.max_context_chars)
    exp_client   = OllamaClient(exp_config)

    if EXPERIMENT_STRATEGY == "simple":
        exp_strategy = SimpleRAGStrategy(exp_client, PromptBuilder(EXPERIMENT_TEMPLATE), exp_preparer)
    else:
        exp_strategy = RefineRAGStrategy(exp_client, exp_preparer)

    exp_result = exp_strategy.generate(EXPERIMENT_QUERY, exp_retrieval)

    print(f"Query    : {EXPERIMENT_QUERY!r}")
    print(f"Strategie: {EXPERIMENT_STRATEGY}  |  Template: {EXPERIMENT_TEMPLATE.name}")
    print(f"Top-k    : {EXPERIMENT_TOP_K}  |  max_context_chars: {EXPERIMENT_MAX_CONTEXT}")
    print()
    print(format_result_summary(exp_result))
    if len(exp_result.answer) > 120:
        print()
        print("Vollständige Antwort:")
        print(exp_result.answer.strip())
else:
    print("Ollama nicht verfügbar, Experiment übersprungen.")

## 19. Zusammenfassung: Architekturentscheidungen und ihre Konsequenzen

### Was in diesem Notebook gezeigt wurde

**Echte Artefakte:** Alle Retrieval-Ergebnisse kamen aus dem persistierten Index, kein
simuliertes Retrieval und keine künstlichen Demo-Chunks.

**Echter Embedder:** Query-Vektoren für Dense-Retrieval wurden vom echten BGE-M3-Modell erzeugt.

**Vollständige Pipeline:** Retrieval, dann `ContextPreparer`, `PromptBuilder`, `OllamaClient`
und `GenerationResult`. Jede Komponente hat eine klare, einzelne Verantwortlichkeit.

### Architekturentscheidungen

| Entscheidung | Gewählt | Nicht gewählt | Konsequenz |
|---|---|---|---|
| Konfiguration | eingefrorenes Dataclass | mutable dict | Reproduzierbarkeit, Prüfpfad |
| HTTP-Client | synchrones `requests` | `aiohttp`, Streaming | Einfachheit, kein Event-Loop |
| Fehlerbehandlung | typisierte Exception-Hierarchie | generisches `Exception` | selektives Retry |
| Context-Budget | zeichen-basiert | Tokenizer | keine schwere Abhängigkeit |
| Dedup-Strategie | erste Instanz erhalten | zufällig oder letzte | Retrieval-Ranking bleibt erhalten |
| Template-Design | versionierte Objekte | Ad-hoc-Strings | Experiment-Tracking |
| SimpleRAG | ein LLM-Aufruf | N Aufrufe | minimale Latenz |
| RefineRAG | N LLM-Aufrufe | ein Aufruf | bessere Integration verteilter Info |
| Ergebnisobjekt | eingefrorenes Dataclass | dict | typsicher, serialisierbar |
| Inference | lokal, CPU | Cloud-API, GPU | Datenschutz, keine Kosten, langsam |

### Der vollständige Datenfluss

Offline, einmalig gebaut: Rohdaten werden über die Ingestion Layer zu `chunks.jsonl`, über die
Embedding Layer zu `embeddings.jsonl` und über die Indexing Layer zu DenseIndex, SparseIndex
und DocumentStore.

Online, pro Query in der Retrieval Layer: Die Query wird über `embed_queries()` zum
Dense-Vektor und über `tokenize()` zu Sparse-Tokens. `DenseIndex.query()` und
`SparseIndex.query()` liefern Kandidaten, die Fusion führt sie zu `RankedCandidate` zusammen,
der DocumentStore-Join macht daraus `RetrievalResult`, und der `BaselineReranker` bewertet die
oberen Ränge nach.

Online, pro Query in der Generation Layer: Aus der Liste der `RetrievalResult` macht
`ContextPreparer.prepare()` bereinigte, deduplizierte und budgetierte Chunks. Daraus baut
`PromptBuilder.build()` den finalen Prompt. `OllamaClient.generate()` ruft das Modell auf, und
`GenerationResult` bündelt Antwort und Messwerte.

### Was in Produktionssystemen anders aussieht

- **Retrieval-Evaluation:** Die Retrieval-Qualität wird explizit gemessen (Recall@K, MRR,
  NDCG), bevor der Generator überhaupt betrachtet wird.
- **Re-Ranker:** Ein zweites, kleineres Modell bewertet die Retrieval-Ergebnisse nach Relevanz,
  bevor sie an den Generator gehen.
- **Prompt-Versionierung:** Templates werden versioniert und mit Experiment-IDs verknüpft.
- **Beobachtbarkeit:** Jeder LLM-Aufruf wird mit Prompt, Parametern, Latenz und Token-Kosten
  protokolliert.
- **Batch-Evaluierung:** Antworten werden gegen einen Evaluationsdatensatz mit automatischen
  Metriken (RAGAS, DeepEval) bewertet statt manuell inspiziert.
- **Kontextfenster-Management:** Produktionssysteme verwenden modellspezifische Tokenizer mit
  exakter Token-Zählung und dynamischer Chunk-Auswahl.

### Was das lokale Setup trotzdem leistet

Das Wesentliche dieses Systems ist nicht die Performance, sondern die Nachvollziehbarkeit. Jede
Komponente ist inspizierbar, jede Entscheidung ist explizit, jede Abwägung ist dokumentiert. Das
ist die Grundlage für fundierte Architekturentscheidungen in komplexeren Systemen.
